# Benchmark for Model 2 (CAPRLM): does context help, and does it beat the baselines?

Runs on the LLM-generated JSONL dataset in `CAPRLM/CAPRLM Data/` (synthetic text: no real person's
data, so it is safe to keep in the repo and open in Colab). The detection stage (regex rules + spaCy)
and the risk model are exactly the code in `CAPRLM/research/`. Nothing here is trained on a test fold.

| Experiment | Question | How |
|---|---|---|
| **1. Detection ceiling** | How much of the sensitive text can *any* filter find? | Share of labelled sensitive spans the detectors propose at all: regex only vs regex + spaCy, per category |
| **2. Feature ablations** | Does *context* (app / domain / title / surrounding words) actually help? | Same model, feature groups switched on one at a time, plus "full minus app/domain/title". Repeated k-fold; mean +/- std |
| **3. Unseen context** | Does it generalise to a kind of screen it never saw? | Leave-one-context-kind-out: train without banking, test on banking; then health; ... |
| **4. Per category** | Where does CAPRLM win or lose against the baselines? | Recall per entity category, summed over folds |

**Read this before quoting any number.** The labels were written by an LLM from a tier-definition
prompt and have **not been audited by a person**. Until a sample is audited, every result is indicative:
it shows the benchmark works and where the model is weak, not that CAPRLM beats anything. The tier
definitions and the label protocol are the researcher's decision (`research/README.md`).

## 0. Settings

In [ ]:
REPO_URL = "https://github.com/imethsuka/J26-IT-308.git"
REPO_BRANCH = "main"
CAPRLM_SUBDIR = "CAPRLM"

# None = "<repo>/CAPRLM/CAPRLM Data". Set a path to use data you uploaded or put on Drive instead,
# e.g. "/content/drive/MyDrive/caprlm-data"  (needed if the data folder is not pushed to GitHub).
DATA_DIR = None

N_FOLDS = 5                     # random k-fold: how many folds per seed
SEEDS = (7,)                    # more seeds = steadier numbers but proportionally slower, e.g. (7, 11, 23)
TUNE = True                     # tune C / class weights / none-threshold inside every fold (slower, correct)
RUN_LEAVE_ONE_KIND_OUT = True   # experiment 3
SPACY_MODEL = "en_core_web_sm"

## 1. Code and packages

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    repo = Path("/content/J26-IT-308")
    if repo.exists():                       # pick up anything pushed since this session started
        subprocess.run(["git", "-C", str(repo), "pull", "--quiet", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo)], check=True)
    WORK = Path("/content/work")
else:                                       # running locally: find the repo that contains this notebook
    repo = Path.cwd().resolve()
    while repo != repo.parent and not (repo / CAPRLM_SUBDIR / "research").is_dir():
        repo = repo.parent
    WORK = Path.cwd() / "benchmark-output"
WORK.mkdir(parents=True, exist_ok=True)

CAPRLM_DIR = repo / CAPRLM_SUBDIR
for needed in ("research/eval/benchmark.py", "research/data/llm_dataset.py", "shared/types.py"):
    assert (CAPRLM_DIR / needed).is_file(), f"{needed} not on GitHub yet: commit and push CAPRLM/ first"
sys.path.insert(0, str(CAPRLM_DIR))
DATA = Path(DATA_DIR) if DATA_DIR else CAPRLM_DIR / "CAPRLM Data"
assert DATA.is_dir(), f"no data folder at {DATA}: push 'CAPRLM Data' to GitHub or set DATA_DIR"
print("repo:", repo, "\ndata:", DATA, "\nin colab:", IN_COLAB)

In [ ]:
# Packages: Colab already has numpy/pandas/scikit-learn/matplotlib; spaCy and its model may be missing.
if importlib.util.find_spec("spacy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy", "joblib"], check=True)
try:
    import spacy
    spacy.load(SPACY_MODEL)
except Exception:
    subprocess.run([sys.executable, "-m", "spacy", "download", SPACY_MODEL], check=True)
os.environ["CAPRLM_SPACY_MODEL"] = SPACY_MODEL

# numpy upgraded under an already-loaded numpy breaks later imports; restart once if that happened.
import importlib.metadata, numpy
if importlib.metadata.version("numpy") != numpy.__version__:
    print("numpy was upgraded: restarting the session. Wait for it to reconnect, then Run all again.")
    os.kill(os.getpid(), 9)
print("packages ready")

## 2. Data

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from research.data.llm_dataset import load_dir
from research.detection import Detector, RegexDetector

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

scenarios, report = load_dir(DATA)
print("load report:", report.summary())
print(f"{len(scenarios)} usable rows, {sum(len(s.gold) for s in scenarios)} labelled spans, "
      f"{sum(1 for s in scenarios if not s.gold)} rows with nothing private")
display(pd.DataFrame({
    "rows per context kind": pd.Series(Counter(s.kind for s in scenarios)),
    "OCR-style rows": pd.Series(Counter(s.kind for s in scenarios if s.ctx.source.value == "ocr")),
}).fillna(0).astype(int))
display(pd.DataFrame(Counter((g.category, g.tier) for s in scenarios for g in s.gold).items(),
                     columns=["pair", "n"]).assign(category=lambda d: d.pair.str[0], tier=lambda d: d.pair.str[1])
        .pivot_table(index="category", columns="tier", values="n", fill_value=0).reindex(columns=["none", "low", "medium", "high"]))

## Experiment 1: the detection ceiling
A filter can only protect what the detection stage proposes. This counts, per category, the share of
**sensitive** labelled spans (tier above `none`) that any detector overlaps. It caps every filter that
shares the detection stage, however good the risk model is.

In [ ]:
detector = Detector.default()                     # regex rules + spaCy
regex_only = Detector(ner=None)

def coverage(det):
    hit, tot = Counter(), Counter()
    for s in scenarios:
        cands = det.detect(s.text)
        for g in s.gold:
            if g.tier == "none":
                continue
            i = s.text.find(g.surface)
            tot[g.category] += 1
            hit[g.category] += any(c.start < i + len(g.surface) and i < c.end for c in cands)
    return hit, tot

h1, tot = coverage(regex_only)
h2, _ = coverage(detector)
cov = pd.DataFrame({"sensitive spans": tot, "regex only": {c: h1[c] / tot[c] for c in tot},
                    "regex + spaCy": {c: h2[c] / tot[c] for c in tot}}).sort_values("sensitive spans", ascending=False)
cov.loc["ALL"] = [sum(tot.values()), sum(h1.values()) / sum(tot.values()), sum(h2.values()) / sum(tot.values())]
display(cov.round(3))

## Experiment 2: does context help? (feature ablations)
Same model family (logistic regression), same folds, same detection. Only the feature groups differ:

| Variant | Sees |
|---|---|
| category only | entity category, detector confidence, which detector |
| + source/app/domain/title | the above + uia/ocr, app name, domain and title buckets |
| + surrounding words | the above + the words next to the span |
| full | the above + span shape + document signals (length, keywords) |
| full minus app/domain/title | full, but **without** the app/domain/title context |

The last row is the direct test of the "context-aware" claim: if it scores the same as full, then
app/domain/title context is not what the model is using. The baselines run on the same folds.
When `TUNE` is on, each variant tunes its own regularisation, class weights and none-threshold
from its own training rows only.

In [ ]:
from research.eval.benchmark import (ABLATIONS, FULL, paired_difference, run_benchmark, summarize,
                                     summarize_categories)

rows, cat_rows = run_benchmark(scenarios, detector, k=N_FOLDS, seeds=SEEDS, split="random", tune=TUNE)
summary = pd.DataFrame(summarize(rows)).set_index("name")
show = ["n_folds", "precision", "recall", "f1", "leakage_rate", "reidentification_rate", "semantic_similarity"]
print(f"random {N_FOLDS}-fold x {len(SEEDS)} seed(s); mean over folds")
display(summary[show].round(3))
display(summary[[c for c in summary.columns if c.endswith("_std")]].round(3).rename(columns=lambda c: c.replace("_std", " (std)")))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, metric, title in zip(axes, ("f1", "recall", "leakage_rate"), ("F1 (higher better)", "Recall (higher better)", "Leakage (lower better)")):
    s = summary
    colors = ["#2563EB" if n.startswith("CAPRLM") else "#94A3B8" for n in s.index]
    ax.barh(range(len(s)), s[metric], xerr=s[metric + "_std"], color=colors)
    ax.set_yticks(range(len(s)))
    ax.set_yticklabels(s.index if metric == "f1" else [""] * len(s), fontsize=8)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=10)
plt.suptitle("Random k-fold, mean +/- std over folds (unaudited LLM labels)", fontsize=10)
plt.tight_layout()
plt.show()

### Is each gap bigger than the noise?
Paired per fold: `mean_diff` is `A - B` for the same fold. **If `|mean_diff|` is not clearly larger than
`std`, or A wins only some of the folds, the gap is not distinguishable from noise.** With a few hundred
rows and 5 folds, differences of a couple of F1 points usually are.

In [ ]:
comparisons = [
    (FULL, "static_masking", "Is full CAPRLM better than the best simple baseline?"),
    (FULL, "rule_based", "... than the existing rule-based baseline?"),
    (FULL, "CAPRLM: category only", "Does context + words + shape beat category alone?"),
    (FULL, "CAPRLM: full minus app/domain/title", "Do app/domain/title specifically help?"),
    (FULL, "CAPRLM: full, untuned", "Did tuning help?"),
]
tbl = []
for a, b, q in comparisons:
    for metric in ("f1", "recall", "leakage_rate"):
        d = paired_difference(rows, a, b, metric)
        better = d["wins"] if metric != "leakage_rate" else d["n"] - d["wins"]   # leakage: lower is better
        tbl.append({"question": q, "metric": metric, "A": a, "B": b, "mean_diff (A-B)": d["mean_diff"],
                    "std": d["std"], "A better in": f"{better}/{d['n']} folds"})
display(pd.DataFrame(tbl).round(3))

## Experiment 3: an unseen context
Each fold holds out **one whole context kind** (e.g. all banking rows): the model is trained on the rest
and tested on the kind it has never seen. This is the harder, more honest test of "context-aware":
a model that only memorised kinds will drop here. Kinds with fewer than 20 rows are skipped (too small to score). Folds are small (one kind each), so expect noise.

In [ ]:
if RUN_LEAVE_ONE_KIND_OUT:
    lrows, _ = run_benchmark(scenarios, detector, split="kind", tune=TUNE)
    lsum = pd.DataFrame(summarize(lrows)).set_index("name")
    print("leave-one-context-kind-out; mean over held-out kinds")
    display(lsum[show].round(3))
    per_kind = pd.DataFrame(lrows).pivot_table(index="fold", columns="name", values="f1")
    display(per_kind[[FULL, "CAPRLM: category only", "static_masking", "rule_based"]].round(3).rename_axis("held-out kind"))

## Experiment 4: where does CAPRLM win or lose?
Recall (share of sensitive spans no longer visible in the output) per entity category, summed over
every fold of experiment 2. Categories with no detector, or that the detector misses, show up here.

In [ ]:
cat = pd.DataFrame(summarize_categories(cat_rows))
cat_table = cat.pivot_table(index="category", columns="name", values="recall")
cat_table.insert(0, "sensitive spans", cat.groupby("category")["sensitive"].first())
display(cat_table.sort_values("sensitive spans", ascending=False).round(3))

## What the numbers say
Printed from the results above, so the wording follows the data. A gap is called **within noise**
unless the mean paired difference is at least twice its std **and** points the same way in most folds.
Even a "clear gap" only means "not fold-to-fold noise": it can still come from the labels.

In [ ]:
def verdict(a, b, metric="f1", higher_is_better=True):
    d = paired_difference(rows, a, b, metric)
    if not d["n"]:
        return f"{a} vs {b}: no data"
    diff = d["mean_diff"] if higher_is_better else -d["mean_diff"]
    better_folds = d["wins"] if higher_is_better else d["n"] - d["wins"]
    consistent = better_folds > d["n"] / 2 if diff > 0 else (d["n"] - better_folds) > d["n"] / 2
    clear = abs(d["mean_diff"]) >= 2 * d["std"] and consistent
    direction = "better" if diff > 0 else "worse"
    return (f"{metric}: {a} is {direction} than {b} by {abs(d['mean_diff']):.3f} "
            f"(std {d['std']:.3f}; better in {better_folds}/{d['n']} folds) -> {'a clear gap' if clear else 'WITHIN NOISE'}")

for a, b in ((FULL, "static_masking"), (FULL, "CAPRLM: category only"), (FULL, "CAPRLM: full minus app/domain/title")):
    print(verdict(a, b, "f1"))
    print(verdict(a, b, "recall"))
    print(verdict(a, b, "leakage_rate", higher_is_better=False))
    print()
print("Reminder: LLM-written, unaudited labels. A clear gap here is a lead to check, not a result.")

## Save the results
Only numbers are written (no text from the dataset): one CSV per experiment. Paste them into the
report or compare with later runs after the data or the model changes.

In [ ]:
stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d-%H%M")
outputs = {
    f"caprlm-ablation-folds-{stamp}.csv": pd.DataFrame(rows),
    f"caprlm-ablation-summary-{stamp}.csv": summary.reset_index(),
    f"caprlm-category-recall-{stamp}.csv": cat,
    f"caprlm-detection-coverage-{stamp}.csv": cov.reset_index(names="category"),
}
if RUN_LEAVE_ONE_KIND_OUT:
    outputs[f"caprlm-leave-one-kind-out-{stamp}.csv"] = pd.DataFrame(lrows)
for name, frame in outputs.items():
    frame.to_csv(WORK / name, index=False)
    print("wrote", WORK / name)
if IN_COLAB:
    from google.colab import files
    for name in outputs:
        files.download(str(WORK / name))